# 🏙️ Air Quality Monitoring — Nostradamus IoTO SDK

Urban air-quality network: four stations (downtown, industrial, highway and park) with distinct pollution profiles report PM2.5, PM10, NO2, O3, CO, SO2, meteorology and a derived AQI index.

This notebook mirrors `examples/smart_city/air_quality_monitoring.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **4 stations × 48 half-hourly readings** (192 records) to the collection
`air_quality_stations`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `air_quality_stations` collection with a schema
2. Generate and send 4 stations × 48 half-hourly readings
3. Query unhealthy PM2.5, ozone peaks, stagnation events and AQI ranking
4. Aggregate PM2.5, AQI and wind-speed statistics
5. Delete a 12-hour window for `AQ_STATION_DOWNTOWN`
6. Delete the collection

## What this notebook highlights

- Station-specific pollution profiles
- Rush-hour traffic multipliers on PM/NO2/CO
- Stagnation detection (high PM10 with low wind)

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🏙️ Nostradamus IoTO SDK — Air Quality Monitoring")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the station collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/smart_city/air_quality_monitoring.py`.

In [ ]:
COLLECTION_NAME = "air_quality_stations"

COLLECTION_SCHEMA = {
                    "key": "AQ_STATION_DOWNTOWN",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "pm25_ugm3": 25.3,
                    "pm10_ugm3": 45.0,
                    "no2_ppb": 35.2,
                    "o3_ppb": 42.0,
                    "co_ppm": 1.5,
                    "so2_ppb": 8.0,
                    "temperature_c": 22.0,
                    "humidity_pct": 55.0,
                    "wind_speed_ms": 3.5,
                    "aqi_index": 65.0,
                    "battery_pct": 92.0,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Urban air quality monitoring network",
            tags=["air-quality", "smart-city", "environment"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send air-quality data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_air_quality_data()` is the
same generator used by the script.

In [ ]:
STATION_PROFILES = {
    "AQ_STATION_DOWNTOWN": {
        "pm25": (15, 60),
        "pm10": (20, 80),
        "no2": (20, 70),
        "o3": (15, 50),
        "co": (0.5, 3.0),
        "so2": (3, 15),
    },
    "AQ_STATION_INDUSTRIAL": {
        "pm25": (30, 120),
        "pm10": (50, 180),
        "no2": (15, 60),
        "o3": (10, 40),
        "co": (0.5, 2.5),
        "so2": (10, 40),
    },
    "AQ_STATION_PARK": {
        "pm25": (5, 30),
        "pm10": (10, 40),
        "no2": (5, 25),
        "o3": (20, 70),
        "co": (0.1, 1.0),
        "so2": (1, 8),
    },
    "AQ_STATION_HIGHWAY": {
        "pm25": (20, 80),
        "pm10": (30, 100),
        "no2": (40, 120),
        "o3": (10, 45),
        "co": (1.0, 5.0),
        "so2": (2, 12),
    },
}

def generate_air_quality_data(num_readings=48):
    """Generate air quality sensor data with station-specific profiles.

    Args:
        num_readings: Readings per station (30-min intervals)

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)

    for station_id, profile in STATION_PROFILES.items():
        for i in range(num_readings):
            timestamp = base_time + timedelta(minutes=30 * i)
            hour = timestamp.hour

            is_rush = hour in (7, 8, 9, 17, 18, 19)
            traffic_factor = 1.3 if is_rush else 1.0

            pm25 = random.uniform(*profile["pm25"]) * traffic_factor
            pm10 = random.uniform(*profile["pm10"]) * traffic_factor
            no2 = random.uniform(*profile["no2"]) * traffic_factor
            o3 = random.uniform(*profile["o3"])
            co = random.uniform(*profile["co"]) * traffic_factor
            so2 = random.uniform(*profile["so2"])

            temp = 15 + 10 * max(0, 1 - abs(hour - 14) / 8.0)
            temp += random.uniform(-3, 3)

            aqi = max(
                pm25 / 55 * 100,
                pm10 / 150 * 100,
                no2 / 100 * 100,
                o3 / 70 * 100,
            )

            reading = {
                "key": station_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "pm25_ugm3": round(min(150, pm25), 1),
                "pm10_ugm3": round(min(200, pm10), 1),
                "no2_ppb": round(min(120, no2), 1),
                "o3_ppb": round(min(80, o3), 1),
                "co_ppm": round(min(5.0, co), 2),
                "so2_ppb": round(min(40, so2), 1),
                "temperature_c": round(min(35, max(5, temp)), 1),
                "humidity_pct": round(random.uniform(30, 90), 1),
                "wind_speed_ms": round(random.uniform(0, 15), 1),
                "aqi_index": round(min(200, max(20, aqi)), 0),
                "battery_pct": round(
                    100 - (i / num_readings) * 25 + random.uniform(-3, 3),
                    1,
                ),
            }
            data.append(reading)

    return data

data = generate_air_quality_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🏭 Industrial station readings**
- **😷 Unhealthy PM2.5 (> 55 µg/m³)**
- **🌫️ Ozone peak (> 60 ppb)**
- **🌬️ Stagnation (PM10 > 80 µg/m³ and wind < 3 m/s)**
- **Top 5 AQI readings**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🏭 Industrial station readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "AQ_STATION_INDUSTRIAL"},
        ],
    )
    summary(result, "🏭 Industrial station readings")

    # 😷 Unhealthy PM2.5 (> 55 µg/m³)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "pm25_ugm3", "operator": "gt", "property_value": 55},
        ],
    )
    summary(result, "😷 Unhealthy PM2.5 (> 55 µg/m³)")

    # 🌫️ Ozone peak (> 60 ppb)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "o3_ppb", "operator": "gt", "property_value": 60},
        ],
    )
    summary(result, "🌫️ Ozone peak (> 60 ppb)")

    # 🌬️ Stagnation (PM10 > 80 µg/m³ and wind < 3 m/s)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "pm10_ugm3", "operator": "gt", "property_value": 80},
            {"property_name": "wind_speed_ms", "operator": "lt", "property_value": 3},
        ],
    )
    summary(result, "🌬️ Stagnation (PM10 > 80 µg/m³ and wind < 3 m/s)")

    # Top 5 AQI readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "aqi_index", "pm25_ugm3"],
        order_by='{"field": "aqi_index", "order": "desc"}',
        limit=5,
    )
    print("🏆 Top 5 AQI readings")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average PM2.5 (µg/m³)", "pm25_ugm3", StatOperation.AVG),
        ("Maximum AQI", "aqi_index", StatOperation.MAX),
        ("Minimum wind speed (m/s)", "wind_speed_ms", StatOperation.MIN),
        ("Distinct stations", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one station

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`AQ_STATION_DOWNTOWN` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "AQ_STATION_DOWNTOWN"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")